In [1]:
%matplotlib notebook

In [2]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import jax
import jax.numpy as jnp
import numpy as onp
from tqdm import tqdm
import matplotlib.pyplot as plt

from msmjax.kernels import split_one_over_r_kernel, SofteningFunctionOneOverR
from msmjax.shortrange import make_compute_U_zero_with_neighborlist, make_compute_f_zero_with_neighborlist, make_compute_U_and_f_zero_with_neighborlist
from msmjax.gridops_multidim import set_up_grids_all_levels, set_up_grid_axis
from msmjax.gridops_multidim import BSplineInterpolationGrid, BSplineInterpolationAxis
from msmjax.gridops_multidim import create_compute_U_oneplus, create_compute_U_and_f_oneplus

import sys

sys.path.append("/home/florian/PhD/work/code/msm_for_nn/")

from msmfornn.splines.nesting import compute_J_zeroplus
from msmfornn.gridtools import construct_grids_all_levels
from msmfornn.splines.coefficients import compute_coeffs_withtruncation
from msmfornn.grid_to_grid_mapping import \
    compute_kernel_stencils_all_gridlevels
from msmfornn.helpers.algoparam_choice import suggest_p, suggest_max_gridlevel_nonPBC


# Helper functions

## Reference

In [3]:
def make_compute_U_zero_reference(kernels, periodic: bool, box_lengths=None):
    k_0 = kernels[0]
    sum_of_higher_kernels_at_zero = jnp.sum(
        jnp.asarray([k(0.0) for k in kernels[1:]])
    )
    
    if periodic and box_lengths is None:
        raise ValueError("`box_lengths` are required in periodic case.")
    
    def compute_U_zero_reference(positions, charges):
        R_ij = positions[:, jnp.newaxis, :] - positions
        if periodic:
            R_ij -= jnp.rint(R_ij / box_lengths) * box_lengths
        qi_qj = charges[:, jnp.newaxis] * charges
        indices_triu = jnp.triu_indices(positions.shape[0], k=1)
        r_ij_triu = jnp.linalg.norm(R_ij[indices_triu], axis=1)
        qi_qj_triu = qi_qj[indices_triu]
        
        pair_term = (qi_qj_triu * jax.vmap(k_0)(r_ij_triu)).sum()
        self_energy_term = 0.5 * jnp.diag(qi_qj).sum() * sum_of_higher_kernels_at_zero
    
        return pair_term - self_energy_term
    
    return compute_U_zero_reference

In [4]:
@jax.jit
def calc_total_e_ref(positions, charges):
    R_ij = positions[:, jnp.newaxis, :] - positions
    qi_qj = charges[:, jnp.newaxis] * charges
    indices_triu = jnp.triu_indices(positions.shape[0], k=1)
    r_ij_triu = jnp.linalg.norm(R_ij[indices_triu], axis=1)
    qi_qj_triu = qi_qj[indices_triu]
    
    return (qi_qj_triu * 1. / r_ij_triu).sum()

@jax.jit
def calc_total_f_ref(positions, charges):
    return -jax.grad(calc_total_e_ref, argnums=0)(positions, charges)

@jax.jit
def calc_total_e_and_f_ref(positions, charges):
    value, grad =  jax.value_and_grad(calc_total_e_ref, argnums=0)(positions, charges)
    return value, -grad

## MSM

In [5]:
def make_wrapped_compute_U_zero(kernels, cutoff, box_lengths, pbcs, neighborlist_reference_positions):
    pbcs = jnp.asarray(pbcs)
    if not (jnp.all(pbcs) or jnp.all(~pbcs)):
        raise ValueError("Mixed boundary conditions currently not supported.")
    periodic = pbcs[0]
    
    # Too large box causes the neighbor list functions to throw strange errors.
    # But in the non-periodic case, the box size is actually irrelevant,
    # and we still get the correct result, and no error, by simply
    # specifying a fictitious small box.
    if not periodic:
        box_lengths = jnp.ones_like(box_lengths)
        
    neighbor_fun, compute_U_zero_with_neighborlist = make_compute_U_zero_with_neighborlist(
        kernels=kernels,
        cutoff=cutoff,
        box_lengths=box_lengths,
        pbcs=PBCS,
    )
    neighbor_list = neighbor_fun.allocate(neighborlist_reference_positions)
    
    def wrapped_compute_U_zero(positions, charges):
        updated_neighbor_list = neighbor_fun.update(positions, neighbor_list)
        return compute_U_zero_with_neighborlist(positions, charges, updated_neighbor_list.idx)
    
    return wrapped_compute_U_zero


def make_wrapped_compute_U_and_f_zero(kernels, cutoff, box_lengths, pbcs, neighborlist_reference_positions):
    pbcs = jnp.asarray(pbcs)
    if not (jnp.all(pbcs) or jnp.all(~pbcs)):
        raise ValueError("Mixed boundary conditions currently not supported.")
    periodic = pbcs[0]
    
    # Too large box causes the neighbor list functions to throw strange errors.
    # But in the non-periodic case, the box size is actually irrelevant,
    # and we still get the correct result, and no error, by simply
    # specifying a fictitious small box.
    if not periodic:
        box_lengths = jnp.ones_like(box_lengths)
        
    neighbor_fun, compute = make_compute_U_and_f_zero_with_neighborlist(
        kernels=kernels,
        cutoff=cutoff,
        box_lengths=box_lengths,
        pbcs=PBCS,
    )
    neighbor_list = neighbor_fun.allocate(neighborlist_reference_positions)
    
    def wrapped_compute(positions, charges):
        updated_neighbor_list = neighbor_fun.update(positions, neighbor_list)
        return compute(positions, charges, updated_neighbor_list.idx)
    
    return wrapped_compute

In [6]:
def wrapper_old_compute_J_zeroplus(p):
    return compute_J_zeroplus(p)

def wrapper_old_construct_kernel_stencils(
    kernels,
    box_lengths,
    level_one_gridspacing,
    level_zero_cutoff,
    n_levels,
    p,
    mu,
):
    omega, _ = compute_coeffs_withtruncation(p=p, mu=mu)
    omega_zeroplus = omega[len(omega) // 2 :]
    grids_oldmsm = construct_grids_all_levels(
        min_pos=onp.zeros_like(box_lengths),
        max_pos=box_lengths,
        p=p,
        level_one_gridspacing=level_one_gridspacing,
        max_gridlevel=n_levels,
    )
    kernel_stencils_nonnegative = compute_kernel_stencils_all_gridlevels(
        kernelfunctions=kernels,
        grids=grids_oldmsm,
        level_zero_cutoff=level_zero_cutoff,
        omega_zeroplus=omega_zeroplus,
    )
    kernel_stencils = [None]
    for stncl in kernel_stencils_nonnegative[1:]:
        pw = [(s - 1, 0) for s in stncl.shape]
        stncl_symm = jnp.pad(stncl, pad_width=pw, mode="reflect")
        kernel_stencils.append(stncl_symm)

    return kernel_stencils


In [7]:
def make_compute_U_oneplus(
    kernels,
    level_one_gridspacing,
    level_zero_cutoff,
    n_levels,
    box_lengths,
    pbcs,
    p,
    mu,
    convolution_methods=None,
):
    n_dim = len(pbcs)

    J_zeroplus = wrapper_old_compute_J_zeroplus(p)
    grids = set_up_grids_all_levels(
        box_lengths=box_lengths,
        level_one_spacings=[level_one_gridspacing] * n_dim,
        pbcs=pbcs,
        n_levels=n_levels,
        p=p,
        J_zeroplus=J_zeroplus,
    )
    kernel_stencils = wrapper_old_construct_kernel_stencils(
        kernels=kernels,
        box_lengths=box_lengths,
        level_one_gridspacing=level_one_gridspacing,
        level_zero_cutoff=level_zero_cutoff,
        n_levels=n_levels,
        p=p,
        mu=mu,
    )

    calculate = create_compute_U_oneplus(
        grids=grids,
        kernel_stencils=kernel_stencils,
        convolution_methods=convolution_methods,
    )

    return calculate


In [8]:
def set_up_msm(
    level_one_gridspacing,
    level_zero_cutoff,
    box_lengths,
    pbcs,
    neighborlist_reference_positions,
    n_levels=None,  # TODO: determine automatically?
    p=None,  # TODO: determine automatically?
    mu=None,  # TODO: determine automatically?
    conv_meth=None,
    **neighbor_kwargs,
):
    pbcs = jnp.asarray(pbcs)
    if pbcs.any():
        raise ValueError(
            "Periodic or mixed boundary conditions currently not supported."
        )

    alpha = level_zero_cutoff / level_one_gridspacing
    if p is None:
        p = suggest_p(alpha)

    # TODO: mu
    # See section "1. Preprocessing" of the article
    if mu is None:
        mu = max(int(4 * alpha + p // 2), 3 * p // 2)

    # TODO: n_levels
    if n_levels is None:
        n_levels = suggest_max_gridlevel_nonPBC(
            min_pos=onp.zeros_like(box_lengths),
            max_pos=box_lengths,
            nb_particles=neighborlist_reference_positions.shape[0],
            level_one_gridspacing=level_one_gridspacing,
            level_zero_cutoff=level_zero_cutoff,
            p=p,
        )
        
    if conv_meth is None:
        convolution_methods = None
    else:
        convolution_methods = [None] + [conv_meth] * n_levels

    kernels = split_one_over_r_kernel(
        max_level=n_levels,
        level_zero_cutoff=level_zero_cutoff,
        softening_function=SofteningFunctionOneOverR(p),
    )
    wrapped_calc_U_zero = make_wrapped_compute_U_zero(
        kernels=kernels,
        cutoff=level_zero_cutoff,
        box_lengths=box_lengths,
        pbcs=pbcs,
        neighborlist_reference_positions=neighborlist_reference_positions,
        **neighbor_kwargs,
    )
    calc_U_oneplus = make_compute_U_oneplus(
        kernels=kernels,
        level_one_gridspacing=level_one_gridspacing,
        level_zero_cutoff=level_zero_cutoff,
        n_levels=n_levels,
        box_lengths=box_lengths,
        pbcs=pbcs,
        p=p,
        mu=mu,
        convolution_methods=convolution_methods,
    )

    def calculate(positions, charges):
        return wrapped_calc_U_zero(positions, charges) + calc_U_oneplus(
            positions, charges
        )
    
    # TODO
    # info = {
    #     "grids": grids, # TODO: return from make_compute_U_oneplus?
    #     "kernel_stencils": kernel_stencils, # TODO: return from make_compute_U_oneplus?
    #     "n_levels": n_levels,
    #     "p": p,
    #     "mu": mu,
    # }

    # return calculate, info    # TODO
    
    return calculate


# Benchmark

In [ ]:
def draw_random_particle_configuration(n_particles, avg_interparticle_distance, n_dim):
    side_length = n_particles ** (1. / n_dim) * avg_interparticle_distance
    box_lengths = jnp.array([side_length] * n_dim)
    pos = rng.uniform(low=onp.zeros_like(box_lengths), high=box_lengths, size=(n_particles, n_dim))
    chg = rng.uniform(low=-1.0, high=1.0, size=N_PARTICLES)
    
    return jnp.array(pos), jnp.array(chg), box_lengths

## Global settings

In [9]:
# Basic geometry
AVG_NEIGHBOR_DISTANCE = 2.5
N_DIM = 3
PBCS = [False] * N_DIM

# MSM
LEVEL_ONE_GRIDSPACING = AVG_NEIGHBOR_DISTANCE
ALPHA = 4.0
LEVEL_ZERO_CUTOFF = ALPHA * LEVEL_ONE_GRIDSPACING
P = 4
MU = 4

## Particle setup

In [41]:
import time

In [49]:
t_1 = time.time()

In [50]:
t_2 = time.time()

In [70]:
SEED = 54
N_STRUCTURES_PER_SIZE = 100

rng = onp.random.default_rng(SEED)

def draw_random_particle_configuration(n_particles, avg_interparticle_distance, n_dim):
    side_length = n_particles ** (1. / n_dim) * avg_interparticle_distance
    box_lengths = jnp.array([side_length] * n_dim)
    pos = rng.uniform(low=onp.zeros_like(box_lengths), high=box_lengths, size=(n_particles, n_dim))
    chg = rng.uniform(low=-1.0, high=1.0, size=N_PARTICLES)
    
    return jnp.array(pos), jnp.array(chg), box_lengths


N_PARTICLES = 2000

pos_initial, chg_initial, box_lengths = draw_random_particle_configuration(
    N_PARTICLES, AVG_NEIGHBOR_DISTANCE, N_DIM
)
calc_total_e_msm = set_up_msm(
    level_one_gridspacing=LEVEL_ONE_GRIDSPACING,
    level_zero_cutoff=LEVEL_ZERO_CUTOFF,
    box_lengths=box_lengths,
    pbcs=PBCS,
    neighborlist_reference_positions=pos_initial,
)
calc_total_e_msm = jax.jit(calc_total_e_msm)

# call once to trigger jit
calc_total_e_msm(pos_initial, chg_initial).block_until_ready()
calc_total_e_ref(pos_initial, chg_initial).block_until_ready()

timings_msm_this_size = []
timings_ref_this_size = []

for iteration_number in range(N_STRUCTURES_PER_SIZE):
    pos, chg, _ = draw_random_particle_configuration(N_PARTICLES, AVG_NEIGHBOR_DISTANCE, N_DIM)
    jax.device_put(pos)
    jax.device_put(chg)    

    t_1 = time.time()
    # e_msm = float(calc_total_e_msm(pos, chg))
    calc_total_e_msm(pos, chg).block_until_ready()
    t_2 = time.time()
    timings_msm_this_size.append(t_2 - t_1)
    
    t_1 = time.time()
    e_ref = float(calc_total_e_ref(pos, chg))
    t_2 = time.time()
    timings_ref_this_size.append(t_2 - t_1)
    
timings_msm_this_size = onp.array(timings_msm_this_size)
timings_ref_this_size = onp.array(timings_ref_this_size)

# print("ref:", calc_total_e_ref(pos, chg))
# print("msm:", calculate_msm_energy(pos, chg))

Suggested value for max grid level L was found according to criterion/criteria:
(if multiple criteria listed, that means they agree on the suggested L)
*) Highest-level grid not coarser than simulation box size ("criterion 1.")
*) Cutoff at highest-but-one level not too large relative to simulation box size ("criterion 2.")


/home/florian/anaconda3/envs/msmjax-dev/lib/python3.8/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int64 to dtype=int32. In future JAX releases this will result in an error.
  warnings.warn("scatter inputs have incompatible types: cannot safely cast "


In [12]:
# calculate_msm_energy = set_up_msm(
#     level_one_gridspacing=LEVEL_ONE_GRIDSPACING,
#     level_zero_cutoff=LEVEL_ZERO_CUTOFF,
#     box_lengths=box_lengths,
#     pbcs=PBCS,
#     neighborlist_reference_positions=pos,
#     conv_meth="scipy-fft",
# )
# calculate_msm_energy = jax.jit(calculate_msm_energy)
# 
# print("ref:", calc_total_e_ref(pos, chg))
# print("msm:", calculate_msm_energy(pos, chg))

In [14]:
jax.device_put(pos)
jax.device_put(chg)

print("Reference:")
%timeit calc_total_e_ref(pos, chg).block_until_ready()
print()

print("MSM:")
%timeit calculate_msm_energy(pos, chg).block_until_ready()

Reference:
2.17 ms ± 22.8 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)

MSM:
11.9 ms ± 26.6 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
